In [ ]:
# !pip install transformers, AutoTokenizer, torch

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [3]:
# 1. Load a text dataset (we use a small example dataset for demonstration)
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")  # raw text WikiText-2
print(f"Number of lines in dataset: {len(dataset)}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Number of lines in dataset: 36718


In [4]:
# 2. Initialize a tokenizer (we'll use GPT-2's tokenizer for compatibility with a GPT-2 model)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a pad by default

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
# 3. Tokenize the dataset efficiently using `.map` with batched processing
def tokenize_function(examples):
    return tokenizer(examples["text"], return_special_tokens_mask=False)

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
# The dataset now has columns like 'input_ids' and 'attention_mask'

print(tokenized_ds[0]["input_ids"][:20])  # print first 20 token IDs of first example for sanity check

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

[]


In [6]:
# 4. Slice into training sequences of fixed length
# For language model training, often we concatenate all texts then split into blocks of e.g. 128 or 512 tokens.
block_size = 128
def group_texts(examples):
    # Concatenate each field
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks = sum(examples["attention_mask"], [])

    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks = concatenated_masks[:total_len]

    # Split into chunks
    result_input_ids = [concatenated_inputs[i:i+block_size] for i in range(0, total_len, block_size)]
    result_masks = [concatenated_masks[i:i+block_size] for i in range(0, total_len, block_size)]

    return {"input_ids": result_input_ids, "attention_mask": result_masks}


lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=1000)
print(f"LM training sequences: {len(lm_ds)}")

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

LM training sequences: 18667


In [7]:
# 5. Create a DataLoader for the tokenized, grouped dataset
# We'll use a custom collate to dynamically pad sequences (though all are same length here by construction)
def collate_fn(batch):
    # Since our sequences are fixed length after grouping, we might just stack them.
    # If they weren't fixed, we could use tokenizer.pad to pad to max length in batch.
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    # For language modeling, labels are the input_ids shifted by one, but
    # Transformers' CausalLM models usually handle that internally if we provide labels = input_ids.
    return {"input_ids": input_ids, "labels": input_ids.clone()}

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)

In [8]:
# 6. Iterate through a couple of batches to see that it works
for batch in train_loader:
    print(batch["input_ids"].shape, batch["labels"].shape)
    break

torch.Size([8, 128]) torch.Size([8, 128])


   ## My Additions
   ### Addition 1: Faster concatenation using itertools.chain

In [9]:
import time
import itertools

# Faster version: same logic as group_texts, but joins lists with itertools.chain
# instead of sum(), which re-copies the growing list every time it adds one
def group_texts_fast(examples):
    concatenated_inputs = list(itertools.chain.from_iterable(examples["input_ids"]))
    concatenated_masks = list(itertools.chain.from_iterable(examples["attention_mask"]))

    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks = concatenated_masks[:total_len]

    result_input_ids = [concatenated_inputs[i:i+block_size] for i in range(0, total_len, block_size)]
    result_masks = [concatenated_masks[i:i+block_size] for i in range(0, total_len, block_size)]
    return {"input_ids": result_input_ids, "attention_mask": result_masks}

# Time the original (load_from_cache_file=False forces a real re-run, not a cached result)
start = time.perf_counter()
lm_ds_original = tokenized_ds.map(group_texts, batched=True, batch_size=1000, load_from_cache_file=False)
original_time = time.perf_counter() - start

# Time the faster version
start = time.perf_counter()
lm_ds_fast = tokenized_ds.map(group_texts_fast, batched=True, batch_size=1000, load_from_cache_file=False)
fast_time = time.perf_counter() - start

print(f"Original (sum):            {original_time:.2f} s")
print(f"Faster (itertools.chain):  {fast_time:.2f} s")
print(f"Speedup:                   {original_time / fast_time:.1f}x")

# Make sure the faster version gives the exact same result
print("Same number of sequences:", len(lm_ds_original) == len(lm_ds_fast))
print("Identical output:", lm_ds_original[:]["input_ids"] == lm_ds_fast[:]["input_ids"])

Map: 100%|##########| 36718/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Original (sum):            11.68 s
Faster (itertools.chain):  3.22 s
Speedup:                   3.6x
Same number of sequences: True
Identical output: True


### Addition 2: Removing blank lines before tokenizing

In [10]:
import time

# 1. Count blank lines (empty, or only spaces/newlines)
num_blank = sum(1 for t in dataset["text"] if t.strip() == "")
print(f"Total lines:          {len(dataset)}")
print(f"Blank lines:          {num_blank} ({num_blank / len(dataset) * 100:.1f}%)")

# 2. Remove them
clean_dataset = dataset.filter(lambda x: x["text"].strip() != "")
print(f"Lines after cleaning: {len(clean_dataset)}")

# 3. Tokenize original vs cleaned, timing both
start = time.perf_counter()
tok_original = dataset.map(tokenize_function, batched=True, remove_columns=["text"], load_from_cache_file=False)
orig_tok_time = time.perf_counter() - start

start = time.perf_counter()
tok_clean = clean_dataset.map(tokenize_function, batched=True, remove_columns=["text"], load_from_cache_file=False)
clean_tok_time = time.perf_counter() - start

# 4. Cut cleaned data into 128-token pieces (using the faster function from Addition 1)
lm_clean = tok_clean.map(group_texts_fast, batched=True, batch_size=1000, load_from_cache_file=False)

print(f"\nTokenizing time  - original: {orig_tok_time:.2f} s | cleaned: {clean_tok_time:.2f} s")
print(f"Training pieces  - original: {len(lm_ds_fast)} | cleaned: {len(lm_clean)}")
print(f"First line before cleaning: {tokenized_ds[0]['input_ids'][:20]}")
print(f"First line after cleaning:  {tok_clean[0]['input_ids'][:20]}")

Total lines:          36718
Blank lines:          12951 (35.3%)


Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Lines after cleaning: 23767


Map: 100%|##########| 36718/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]


Tokenizing time  - original: 13.22 s | cleaned: 9.45 s
Training pieces  - original: 18667 | cleaned: 18676
First line before cleaning: []
First line after cleaning:  [796, 569, 18354, 7496, 17740, 6711, 796, 220, 198]


### Addition 3: Comparing block sizes (128 vs 256 vs 512 vs 1024)

In [11]:
# Same as group_texts_fast, but the piece size can be changed
def group_texts_by_size(examples, size):
    concatenated_inputs = list(itertools.chain.from_iterable(examples["input_ids"]))
    concatenated_masks = list(itertools.chain.from_iterable(examples["attention_mask"]))
    total_len = (len(concatenated_inputs) // size) * size
    return {
        "input_ids": [concatenated_inputs[i:i+size] for i in range(0, total_len, size)],
        "attention_mask": [concatenated_masks[i:i+size] for i in range(0, total_len, size)],
    }

# Total number of tokens in the cleaned data
total_tokens = sum(len(ids) for ids in tok_clean["input_ids"])

# Try each piece size and record the results
results = []
for size in [128, 256, 512, 1024]:
    lm = tok_clean.map(group_texts_by_size, batched=True, batch_size=1000,
                       fn_kwargs={"size": size}, load_from_cache_file=False)
    used = len(lm) * size
    dropped = total_tokens - used
    results.append((size, len(lm), used, dropped, dropped / total_tokens * 100))

# Print a comparison table
print(f"\nTotal tokens in cleaned data: {total_tokens}\n")
print(f"{'Block size':>10} | {'Sequences':>9} | {'Tokens used':>11} | {'Tokens dropped':>14} | {'% dropped':>9}")
print("-" * 66)
for size, n_seq, used, dropped, pct in results:
    print(f"{size:>10} | {n_seq:>9} | {used:>11} | {dropped:>14} | {pct:>8.2f}%")

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]


Total tokens in cleaned data: 2391884

Block size | Sequences | Tokens used | Tokens dropped | % dropped
------------------------------------------------------------------
       128 |     18676 |     2390528 |           1356 |     0.06%
       256 |      9331 |     2388736 |           3148 |     0.13%
       512 |      4660 |     2385920 |           5964 |     0.25%
      1024 |      2325 |     2380800 |          11084 |     0.46%
